## Possible improvements:

- **Query rewriting**: Generate multiple distinct variations of the user's original query. When it comes to semantic search, simply rephrasing a query might retrieve a different set of results. The goal kind of is to encode different views of the same underlying concept.

- **Hybrid search**: Combining lexical and semantic search. Simple keyword search can be used to make a faster and less compute-intensive search across the document set, finding candidates that a vector search alone might not surface.

**RAG Fusion** is an approach that can be used to run retrieval independently for each variation of the query and merge the retrieved rankings using Reciprocal Rank Fusion (RRF). Documents that consistently rank well across multiple rewritten queries receive higher fused scores.

$RRF(d) = \sum_i \frac{1}{k + rank_i(d)}$

In [ ]:
import sys
sys.path.append("..")

from src.utils import load_embed_model, load_embeddings, load_gen_model
from src.pipeline import Generator

In [2]:
import torch

assert torch.cuda.is_available(), "CUDA is not available"

vlm, vlm_processor = load_gen_model()
generator = Generator(vlm, vlm_processor).to("cuda")

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

### Query rewriting

In [ ]:
import json

output_format_prompt = """Return your output as follows:

{"queries": ["<Variation 1>", "<Variation 2>", "<Variation 3>"]}

"""

REWRITE_PROMPT = """You are a helpful assistant that rephrases questions.

Given a question, generate three different variations of the question.

Variation 1: Same exact meaning. Remove conversational language and make sure the question is grammatically correct.
Variation 2: Ask a subquestion that needs to be answered to answer the original question.
Variation 3: Ask a more generic question that gets at the underlying principles needed to answer the original question.

If there are acronyms or words you are not familiar with, DO NOT try to rephrase them.

{output_format_prompt}

Question: {query}

Three variations of the question:
"""

query = "what's the electric one with yellow fur and red cheeks"
output = generator.run_single_user_message(REWRITE_PROMPT.format(query=query, output_format_prompt=output_format_prompt))

try:
    queries = json.loads(output).get("queries")
    assert len(queries) == 3
except:
    queries = [query]


queries

['What is the electric animal with yellow fur and red cheeks?',
 'What character has electric abilities and yellow fur with red cheeks?',
 'What electric creature has yellow fur and red cheeks?']

### Hybrid search

1. semantic search on the rewritten queries
2. lexical search on a single query
3. merge the different rankings using RRF

In [4]:
from pathlib import Path

embeddings_path = Path.cwd().parent / "image_text_embeddings.safetensors"
assert embeddings_path.exists(), "Embeddings file not found"

embed_model = load_embed_model()
embeddings = load_embeddings(embeddings_path)["image_text_embeddings"]

Loading weights:   0%|          | 0/600 [00:00<?, ?it/s]

In [5]:
embed_model.to("cuda")
embeddings = embeddings.to("cuda")

embed_model.device

device(type='cuda', index=0)

In [6]:
from datasets import load_dataset

ds = load_dataset("cian-ps/pokedb-vl-1k", split="train")
ds

Dataset({
    features: ['identifier', 'ndex_id', 'name', 'image', 'markdown_text'],
    num_rows: 1025
})

In [7]:
def _l2_normalize(x: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    return x / (x.norm(p=2, dim=-1, keepdim=True) + eps)

def _retrieve(
    embed_queries: torch.Tensor,
    embed_docs: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    cos_sim = _l2_normalize(embed_queries) @ _l2_normalize(embed_docs).T
    scores, rankings = torch.sort(cos_sim, dim=-1, descending=True)
    return scores, rankings

with torch.inference_mode():
    embed_queries = embed_model.encode_documents(texts=queries)

scores, rankings = _retrieve(embed_queries, embeddings)
scores.shape, rankings.shape

(torch.Size([3, 1025]), torch.Size([3, 1025]))

In [8]:
scores, rankings

(tensor([[0.2539, 0.2422, 0.2383,  ..., 0.0903, 0.0806, 0.0781],
         [0.2910, 0.2891, 0.2871,  ..., 0.1309, 0.1299, 0.1279],
         [0.2441, 0.2432, 0.2432,  ..., 0.0771, 0.0767, 0.0688]],
        device='cuda:0', dtype=torch.bfloat16),
 tensor([[192, 662, 643,  ..., 154, 252, 297],
         [192, 662, 643,  ..., 790, 268, 252],
         [662, 192, 248,  ..., 287, 252, 297]], device='cuda:0'))

In [9]:
from rank_bm25 import BM25Okapi
import numpy as np

text_store = [doc.lower().split() for doc in ds["markdown_text"]]
bm25 = BM25Okapi(text_store)

def _bm25_search(query):
    query_tokens = query.lower().split()
    scores = bm25.get_scores(query_tokens)
    ranks = np.argsort(scores, descending=True)
    return scores[ranks], ranks

bm25_scores, bm25_ranks = _bm25_search(query)
bm25_scores, bm25_ranks, ds.select(bm25_ranks[:5])["identifier"]

(array([18.63156788, 17.37141408, 16.60659592, ...,  7.54990217,
         7.53217514,  7.50574966], shape=(1025,)),
 array([656, 624, 436, ...,  10, 912, 455], shape=(1025,)),
 Column(['pichu-default', 'orbeetle-default', 'jolteon-default', 'oricorio-baile', 'magby-default']))

In [10]:
def rrf_merge(r_semantic: torch.Tensor, r_lexical: np.ndarray | None = None) -> tuple[np.ndarray, np.ndarray]:
    k_constant = 60  # The value of 60 is a standard academic convention
    rankings = r_semantic.cpu().numpy()
    if r_lexical is not None:
        rankings = np.vstack((rankings, r_lexical))
    rrf_scores = np.zeros(rankings.shape[1])
    for r in rankings:
        for rank, idx in enumerate(r):
            rrf_scores[idx] += 1 / (k_constant + rank + 1)
    merged_ranking = np.argsort(rrf_scores, descending=True)
    return rrf_scores[merged_ranking], merged_ranking

rrf_scores, merged_ranking = rrf_merge(rankings, bm25_ranks)
rrf_scores, merged_ranking

(array([0.06255457, 0.05952107, 0.0557471 , ..., 0.00385376, 0.00378599,
        0.00378168], shape=(1025,)),
 array([656, 662, 643, ..., 162, 102, 100], shape=(1025,)))

In [11]:
top_k = 5

top_k_docs = ds.select(merged_ranking[:top_k])
top_k_docs["identifier"]

Column(['pichu-default', 'pikachu-default', 'pawmi-default', 'dedenne-default', 'togedemaru-default'])

Putting it all together:

In [12]:
def rag_fusion(query, query_im=None):
    output = generator.run_single_user_message(REWRITE_PROMPT.format(query=query, output_format_prompt=output_format_prompt))
    try:
        queries = json.loads(output).get("queries")
        assert len(queries) == 3
    except:
        queries = [query]
    
    with torch.inference_mode():
        if query_im is None:
            embed_queries = embed_model.encode_documents(texts=queries)
        else:
            embed_queries = embed_model.encode_documents(images=[query_im] * len(queries), texts=queries)

    _, rankings = _retrieve(embed_queries, embeddings)
    _, bm25_ranks = _bm25_search(queries[0])
    semantic_rrf_scores, semantic_ranking = rrf_merge(rankings)
    hybrid_rrf_scores, hybrid_ranking = rrf_merge(rankings, bm25_ranks)


    return {
        "queries": queries,
        "rrf_scores": {"semantic": semantic_rrf_scores, "hybrid": hybrid_rrf_scores},
        "ranking": {"semantic": semantic_ranking, "hybrid": hybrid_ranking},
    }

result = rag_fusion("Who is the strongest pokemon?")
result, ds.select(result["ranking"]["semantic"][:top_k])["identifier"], ds.select(result["ranking"]["hybrid"][:top_k])["identifier"]

({'queries': ['Who is the strongest Pokémon?',
   'What is the strongest Pokémon?',
   'What makes a Pokémon strong?'],
  'rrf_scores': {'semantic': array([0.04839549, 0.04813108, 0.04452313, ..., 0.00277351, 0.00276923,
          0.00276838], shape=(1025,)),
   'hybrid': array([0.05018869, 0.04978438, 0.04734003, ..., 0.00382711, 0.00381094,
          0.00374633], shape=(1025,))},
  'ranking': {'semantic': array([515, 686, 523, ..., 844, 252, 527], shape=(1025,)),
   'hybrid': array([686, 515, 523, ..., 624, 351, 100], shape=(1025,))}},
 Column(['machop-default', 'probopass-default', 'magneton-default', 'aron-default', 'machoke-default']),
 Column(['probopass-default', 'machop-default', 'magneton-default', 'aron-default', 'rayquaza-default']))

### Generation

In [13]:
cot_prompt = """You are having a conversation with the users about different Pokémon species.

Below you are given a list of documents as context and a list of questions.
Different document chunks are separated by three dashes (---).
Different questions are separated by three dashes, too.

Documents:
{documents}

Questions:
{questions}

For each question go through the following steps:
Step 1. Determine if the question is off topic. Remember, the topic is Pokémon.
Step 2. Which parts of the documents directly apply to the question?
Step 3. Can you answer the question based on evidence provided in the documents?
Step 4. Refine your answer with each question.

Generate a final context-rich and factual answer.

Provide only the final answer as your output. DO NOT include your reasoning.
Final Answer:
"""

top_k = 10

top_k_docs = ds.select(result["ranking"]["hybrid"][:top_k])
context = "\n\n---\n\n".join(top_k_docs["markdown_text"])
questions = "\n\n---\n\n".join(result["queries"])

print(generator.run_single_user_message(cot_prompt.format(documents=context, questions=questions)))

The strongest Pokémon is Rayquaza, which has the highest base stat total of 680. It is a Dragon and Flying type with Attack and Sp. Atk both at 150, and its base stat total is the highest among all Pokémon listed.


In [14]:
import requests
from PIL import Image

im_url = "https://img.pokemondb.net/artwork/large/pikachu.jpg"
query_image = Image.open(requests.get(im_url, stream=True).raw)

def cot_generate(query, query_im=None, top_k=10):
    result = rag_fusion(query, query_im)
    context = "\n\n---\n\n".join(ds.select(result["ranking"]["hybrid"][:top_k])["markdown_text"])
    questions = "\n\n---\n\n".join(result["queries"])
    doc_images = ds.select(result["ranking"]["hybrid"][:top_k])["image"]
    return generator.run_single_user_message(cot_prompt.format(documents=context, questions=questions), query_im), doc_images

cot_generate("Whos that cutie?", query_image)[0]

'The cutie is Pikachu.'

In [15]:
im_url = "https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fstatic1.thegamerimages.com%2Fwordpress%2Fwp-content%2Fuploads%2F2025%2F01%2Fpokemon-eevee-eeveelutions.jpeg&f=1&nofb=1&ipt=8f2a156f7ca9c751fa3a105de7c6bad1446ce07378325358ed5fcb9eb622daa4"
query_image = Image.open(requests.get(im_url, stream=True).raw)

response, doc_images = cot_generate("What do you see?", query_image)
response

'What is observed?\n\nThe image shows a collage of various Pokémon, with the central figure being Eevee. The surrounding Pokémon are the evolved forms of Eevee, including Vaporeon, Jolteon, Flareon, Espeon, Umbreon, Leafeon, Glaceon, and Sylveon. The image also includes the original Eevee in the center, surrounded by its evolved forms.\n\nWhat is visible?\n\nThe image displays a collection of Pokémon, including Eevee and its evolved forms. The central figure is Eevee, and the surrounding Pokémon are its evolved forms. The image also includes the original Eevee in the center, surrounded by its evolved forms.\n\nWhat can be seen?\n\nThe image shows a collage of various Pokémon, including Eevee and its evolved forms. The central figure is Eevee, and the surrounding Pokémon are its evolved forms. The image also includes the original Eevee in the center, surrounded by its evolved forms.'

In [16]:
def call_vlm(messages, vlm=vlm, processor=vlm_processor, max_new_tokens=512, temperature=0.1):
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )

    inputs = inputs.to(vlm.device)

    with torch.no_grad():
        output_ids = vlm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.9,
        )
    
    output_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, output_ids)
    ]

    output_text = processor.batch_decode(
        output_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    return output_text[0]

cot_prompt = """Below you are given a list of documents as context and a list of questions.
Different document chunks are separated by three dashes (---).
Different questions are separated by three dashes, too.

Documents:
{documents}

Questions:
{questions}

For each question go through the following steps:
Step 1. What is the question's essential meaning and how does it relate to the documents?
Step 2. Which parts of the documents directly apply to the question?
Step 3. Can you answer the question based on evidence provided in the documents?
Step 4. Refine your answer with each question.

Provide your output as a final summary of your findings. Your output MUST NOT exceed 300 words.

Output:
"""

top_k = 10
result = rag_fusion("What are these pokemon?", query_image)
context = "\n\n---\n\n".join(ds.select(result["ranking"]["hybrid"][:top_k])["markdown_text"])
questions = "\n\n---\n\n".join(result["queries"])

messages = [
    {
        "role": "system",
        "content": [
            {"type": "text", "text": "You are having a conversation with the users about different Pokémon species."},
        ],
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": cot_prompt.format(documents=context, questions=questions)},
            {"type": "image", "image": query_image},
        ],
    }
]

print(result["queries"])
print(list(ds.select(result["ranking"]["hybrid"][:top_k])["identifier"]))


call_vlm(messages)

['What are these Pokémon?', 'What do these Pokémon refer to?', 'What is the meaning of these Pokémon?']
['sylveon-default', 'mew-default', 'eevee-default', 'espeon-default', 'vaporeon-default', 'purugly-default', 'flareon-default', 'skitty-default', 'glameow-default', 'azurill-default']


'Based on the provided documents, the Pokémon listed are the evolution line of Eevee, which includes its various forms and their characteristics. The documents detail the evolution of Eevee into the following species: Vaporeon, Jolteon, Flareon, Espeon, Umbreon, Leafeon, Glaceon, and Sylveon. Each of these Pokémon is described with its type, physical characteristics, base stats, abilities, and evolution path. The documents also provide information on their catch rates, EV yields, and type matchups. The question "What are these Pokémon?" is answered by identifying them as the evolution line of Eevee. The question "What do these Pokémon refer to?" is answered by explaining that they are the various forms of Eevee. The question "What is the meaning of these Pokémon?" is answered by explaining that they are the evolution line of Eevee, which is a key concept in the Pokémon franchise. The documents provide a comprehensive overview of the evolution line of Eevee, including their types, physi

In [17]:
simple_qa_prompt = """
Generate a final concise and factual answer.
Your answer must be directly answering the following question in a conversational setting.

Question: {question}

Answer:
"""

messages.append({
    "role": "assistant",
    "content": [
        {"type": "text", "text": call_vlm(messages)},
    ],
})

messages.append({
    "role": "user",
    "content": [
        {"type": "text", "text": simple_qa_prompt.format(question=result["queries"][0])},
        {"type": "image", "image": query_image},
    ],
})

call_vlm(messages)

'These Pokémon are the Eeveelutions, which are the evolved forms of Eevee. The Eeveelutions are a group of Pokémon that evolve from Eevee, and they are known for their unique abilities and types. The Eeveelutions include Vaporeon, Jolteon, Flareon, Espeon, Umbreon, Leafeon, Glaceon, Sylveon, Mew, and others. Each Eeveelution has a unique appearance and type, and they are all part of the Pokémon franchise.'

In [52]:
def rag_cot_generation(query, query_im=None, top_k=10):
    result = rag_fusion(query, query_im)
    context = "\n\n---\n\n".join(ds.select(result["ranking"]["hybrid"][:top_k])["markdown_text"])
    questions = "\n\n---\n\n".join(result["queries"])

    messages = [
        {
            "role": "system",
            "content": [
                {"type": "text", "text": "You are having a conversation with the users about different Pokémon species."},
            ],
        },
        {
            "role": "user",
            "content": [
                {"type": "text", "text": cot_prompt.format(documents=context, questions=questions)},
            ],
        }
    ]

    if query_im is not None:
        messages[1]["content"].append({"type": "image", "image": query_im})
    
    print(call_vlm(messages))
    
    messages.append({
        "role": "assistant",
        "content": [
            {"type": "text", "text": call_vlm(messages)},
        ],
    })

    messages.append({
        "role": "user",
        "content": [
            {"type": "text", "text": simple_qa_prompt.format(question=result["queries"][0])},
        ],
    })

    if query_im is not None:
        messages[-1]["content"].append({"type": "image", "image": query_im})
    
    return call_vlm(messages)

rag_cot_generation("What pokemon is this?", query_image)

Based on the provided documents, the Pokémon in the image is Eevee. This is determined by analyzing the information in the documents.

Step 1: The question "What is this Pokémon?" asks for the identity of the Pokémon shown in the image. The image displays multiple Pokémon, but the central figure is Eevee, which is the primary subject of the text.

Step 2: The documents contain a detailed description of Eevee, including its National Dex number (#133), its type (Normal), its appearance (brown body color, quadruped body shape), and its evolutionary line. The text also describes its abilities and base stats, which are consistent with the Pokémon in the image.

Step 3: The documents confirm that Eevee is the Pokémon depicted in the image. The text explicitly states that Eevee is the "central" Pokémon in the image, and it is the starting point of the evolutionary line for the other Pokémon shown.

Step 4: The answer is refined by confirming that the Pokémon in the image is Eevee, as it is th

"The Pokémon in the image is Eevee. It's the central figure in the collage, and it's a Normal-type Pokémon."

In [54]:
rag_cot_generation("Which is the pokemon with the highest number of evolutionary forms?")

**Step 1. Essential meaning and relation to documents:**  
The questions ask about the Pokémon with the most evolutionary forms, the greatest number of evolutionary stages, and the highest number of evolutionary paths. These relate to the "Evolution" section in each Pokémon's entry, which lists the stages and conditions for evolution.

**Step 2. Relevant document parts:**  
Each Pokémon entry includes an "Evolution" section detailing its evolutionary line and stages. The number of evolutionary paths is determined by the number of stages in the line, while the number of forms is determined by the number of forms in the line.

**Step 3. Answer based on documents:**  
- **Question 1:** Wurmple has 4 evolutionary stages (Wurmple → Silcoon → Cascoon → Beautifly → Dustox), but it has only one form.  
- **Question 2:** Pikachu has 3 evolutionary stages (Pichu → Pikachu → Raichu → Raichu (Alolan Form)), but it has only one form.  
- **Question 3:** Eevee has 8 evolutionary stages (Eevee → Vapo

'Eevee has the most evolutionary forms, with 8 distinct stages. It evolves into Vaporeon, Jolteon, Flareon, Espeon, Umbreon, Leafeon, Glaceon, and Sylveon. No other Pokémon has more than 8 evolutionary paths.'